# Speckle2Self — Simulator Benchmark Metrics
Evaluates `model_199.pth` (trained on in-vivo peripheral nerve US) on the S2S authors' synthetic test sets (S-I, S-II, S-III) and compares against Table 4 of the paper.

In [1]:
import sys
import numpy as np
from pathlib import Path
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from skimage.feature import graycomatrix, graycoprops
import pandas as pd

try:
    import lpips as lpips_lib
    import torch
    LPIPS_AVAILABLE = True
except ImportError:
    LPIPS_AVAILABLE = False
    print("[WARN] lpips not installed — LPIPS metric will be skipped")

sys.path.insert(0, str(Path(r"C:\P2P\Dissertation\tracked") / "speckle2self"))
from utils.image_ops import linear_normalization

In [2]:
TEST_SETS = {
    "S-I":   r"D:\dissertation-results\s2s_data\simulator\S1\test_data_final.npy",
    "S-II":  r"D:\dissertation-results\s2s_data\simulator\SII\test_data.npy",
    "S-III": r"D:\dissertation-results\s2s_data\simulator\SIII\test_data_final.npy",
}
OUTPUT_FILES = {
    "S-I":   r"D:\dissertation-results\s2s_eval\outputs_SI.npy",
    "S-II":  r"D:\dissertation-results\s2s_eval\outputs_SII.npy",
    "S-III": r"D:\dissertation-results\s2s_eval\outputs_SIII.npy",
}

# Paper Table 4 — S2S authors' model (trained on simulator data)
PAPER_RESULTS = {
    "S-I":   {"PSNR": 18.59, "SSIM": 0.771, "Hom": 0.803, "LPIPS": 0.231},
    "S-II":  {"PSNR": 18.94, "SSIM": 0.777, "Hom": 0.792, "LPIPS": 0.233},
    "S-III": {"PSNR": 22.90, "SSIM": 0.922, "Hom": 0.876, "LPIPS": 0.173},
}

In [3]:
def compute_homogeneity(image_f32):
    img_u8 = (np.clip(image_f32, 0, 1) * 255).astype(np.uint8)
    glcm = graycomatrix(img_u8, distances=[1], angles=[0], levels=256,
                        symmetric=True, normed=True)
    return graycoprops(glcm, "homogeneity")[0, 0]


def compute_metrics(outputs, gt_images, lpips_fn=None):
    psnrs, ssims, homs, lpips_vals = [], [], [], []
    device = next(iter(lpips_fn.parameters())).device if lpips_fn else None
    for out, gt in zip(outputs, gt_images):
        gt_norm = linear_normalization(gt)
        psnrs.append(peak_signal_noise_ratio(gt_norm, out, data_range=1.0))
        ssims.append(structural_similarity(gt_norm, out, data_range=1.0))
        homs.append(compute_homogeneity(out))
        if lpips_fn is not None:
            t_out = (torch.tensor(out).unsqueeze(0).unsqueeze(0)
                     .repeat(1, 3, 1, 1).float().to(device) * 2 - 1)
            t_gt  = (torch.tensor(gt_norm).unsqueeze(0).unsqueeze(0)
                     .repeat(1, 3, 1, 1).float().to(device) * 2 - 1)
            lpips_vals.append(lpips_fn(t_out, t_gt).item())
    result = {"PSNR": np.mean(psnrs), "SSIM": np.mean(ssims), "Hom": np.mean(homs)}
    if lpips_vals:
        result["LPIPS"] = np.mean(lpips_vals)
    return result

In [4]:
lpips_fn = None
if LPIPS_AVAILABLE:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    lpips_fn = lpips_lib.LPIPS(net="alex").to(device)

our_results = {}
for name in TEST_SETS:
    data = np.load(TEST_SETS[name])
    gt      = data[:, 1]
    outputs = np.load(OUTPUT_FILES[name])
    print(f"{name}: {len(outputs)} images")
    our_results[name] = compute_metrics(outputs, gt, lpips_fn)

print("Done.")

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


c:\Users\João\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\João\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: c:\Users\João\AppData\Local\Programs\Python\Python313\Lib\site-packages\lpips\weights\v0.1\alex.pth
S-I: 39 images
S-II: 32 images
S-III: 37 images
Done.


In [5]:
metrics = ["PSNR", "SSIM", "Hom", "LPIPS"]

rows = []
for name in ["S-I", "S-II", "S-III"]:
    row_ours  = {"Set": name, "Model": "Ours (model_199)"}
    row_paper = {"Set": name, "Model": "S2S paper (Table 4)"}
    for m in metrics:
        row_ours[m]  = round(our_results[name].get(m, float("nan")), 4)
        row_paper[m] = PAPER_RESULTS[name].get(m, float("nan"))
    rows.append(row_ours)
    rows.append(row_paper)

df = pd.DataFrame(rows).set_index(["Set", "Model"])
df

PSNR    SSIM     Hom   LPIPS
Set   Model                                               
S-I   Ours (model_199)     18.9167  0.6752  0.7312  0.4852
      S2S paper (Table 4)  18.5900  0.7710  0.8030  0.2310
S-II  Ours (model_199)     19.2005  0.6741  0.7422  0.5008
      S2S paper (Table 4)  18.9400  0.7770  0.7920  0.2330
S-III Ours (model_199)     16.6019  0.6597  0.6749  0.4020
      S2S paper (Table 4)  22.9000  0.9220  0.8760  0.1730

In [6]:
# ── XLSX export ───────────────────────────────────────────────────────────────
from pathlib import Path
from openpyxl.utils import get_column_letter

xlsx_path = Path(r"D:\graphs-n-tables\speckle2self") / 's2s_metrics.xlsx'
with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Simulator')
    ws = writer.sheets['Simulator']
    for col_idx in range(1, ws.max_column + 1):
        col_letter = get_column_letter(col_idx)
        ws.column_dimensions[col_letter].width = max(
            (len(str(c.value)) for c in ws[col_letter] if c.value), default=8
        ) + 3
print(f'Saved → {xlsx_path}')

Saved → D:\graphs-n-tables\speckle2self\s2s_metrics.xlsx
